# BUCLE WHILE
El bucle se ejecuta mientras la condición sea verdadera.

## Sintaxis 
```bash
while [ condición ]; do
    # comandos
done
```

In [7]:
%%bash
contador=1

while [ $contador -le 5 ]
do
    echo "Número: $contador" 
    contador=$(( contador + 1 ))  
done

Número: 1
Número: 2
Número: 3
Número: 4
Número: 5


In [11]:
%%bash
# Usando ((...))

contador=0

while (( contador <= 5 ))
do 
    echo "Número $contador"
    (( contador++ ))
done

# Más legible y común en scripts modernos.


Número 0
Número 1
Número 2
Número 3
Número 4
Número 5


## Formas

### Leer archivos
```bash
while read linea
do 
    echo "Línea: $linea"

done < archivo
```

In [19]:
%%bash

while read linea
do 
    echo "Linea: $linea"
done < ../files/test/orden.txt

Linea: 1
Linea: 2
Linea: 3
Linea: 4
Linea: 5
Linea: 6
Linea: 7
Linea: 8
Linea: 9


### `getopts`
**getopts** es un comando integrado en Bash que permite analizar opciones de línea de comandos en scripts de forma sencilla y eficiente. Funciona en bucles `while` para procesar opciones cortas (de un solo carácter con un guion, como `-a`, `-f`) y sus argumentos asociados.


#### Funcionamiento básico
- Se define una cadena de opciones (`optstring`) que especifica qué opciones son válidas y si requieren un argumento (se añade `:` después de la letra).
- Cada iteración del bucle `while getopts` procesa una opción y la almacena en la variable `opt` (por defecto), mientras que el argumento asociado se guarda en `OPTARG`.
- Si una opción no es válida, `opt` toma el valor `?`, y si una opción que requiere argumento no lo recibe, `opt` es `:`.

#### Variables clave
- `OPTIND`: Índice del próximo argumento no procesado.
- `OPTARG`: Valor del argumento asociado a la opción actual.
- `OPTERR`: Controla si se muestran errores automáticamente (se puede desactivar con `:` al inicio de `optstring`).

#### Estructura mínima
```bash
while getopts "ab:c" opt; do
  case $opt in
    a) echo "Opción -a";;
    b) echo "Opción -b con valor $OPTARG";;
    c) echo "Opción -c";;
    ?) echo "Opción inválida";;
  esac
done
```
Desarmamos el `optstring`
```bash
"ab:c"
```
Significa: 
| Letra | Significado                        |
| ----- | ---------------------------------- |
| a     | flag sin argumento                 |
| b:    | flag **con argumento obligatorio** |
| c     | flag sin argumento                 |


> El `:` es clave: indica que espera un valor.

#### Simulación
```bash
./script.sh -a -b hola -c
```

##### ***Iteración 1 :*** 
`opt = a`

Entra
```bash
opt = a
```
##### ***Iteración 2 :*** 
`opt = b`, `OPTARG = hola`

Entra en:
```bash
b) echo "Opción -b con valor hola"
```


##### ***Iteración 3 :*** 
`opt = c`

Entra en:
```bash
c) echo "Opción -c"
```

#### Usando al inicio `:ab:c`
Se interpreta como:

| Parte        | Significado                           |
| ------------ | ------------------------------------- |
| `:` (inicio) | activa **modo silencioso de errores** |
| `a`          | opción simple                         |
| `b:`         | opción que **requiere argumento**     |
| `c`          | opción simple                         |


Este `:` no es un opción

> Es un modo de comportamiento

| Sin `:` al inicio                    | Con `:` al inicio                           |
| ------------------------------------ | ------------------------------------------- |
| Bash imprime errores automáticamente | Tú manejas errores                          |
| Falta argumento → `opt=?`            | Falta argumento → `opt=:`                   |
| Opción inválida → `opt=?`            | Opción inválida → `opt=?` pero con `OPTARG` |



```bash
while getopts ":ab:c" opt
do
  echo "opt=$OPT, OPTARG=$OPTARG"
done
```
***Caso 1: Todo correcto***
Ejecutar:
```bash
./script.sh -a -b hola -c
```
Iteracciones
```bash
opt=a, OPTARG=
opt=b, OPTARG=hola
opt=c, OPTARG=
```
***Caso 2: Opción inválida***
```bash
./script.sh -x
```
Resultado:
```bash
opt=?, OPTARG=x
```
> Porque usamos : al inicio → Bash NO imprime error, tú lo manejas.

***Caso 3: Falta argumentos en `-b`***
```bash
./script.sh -b
```
Resultado:
```bash
opt=:, OPTARG=b
```
  * `opt=:` -> faltó argumento
  * `OPTARG=b` -> la opción que falló

#### Ejemplo
Versión más completa (manejo de errores fino)
```bash
while getopts ":ab:c" opt; do
  case $opt in
    a) echo "A activado";;
    b) echo "B con valor: $OPTARG";;
    c) echo "C activado";;
    :)
      echo "La opción -$OPTARG requiere un argumento"
      ;;
    ?)
      echo "Opción inválida: -$OPTARG"
      ;;
  esac
done
```